# Model 2: Linear Regression On PCs

This version of the model uses log 2 fold change rather than raw fold change.

In [1]:
from model import Model
import pandas as pd

## Load data

In [2]:
gene_names_list = pd.read_csv('../../data/controls/gene_names.csv')['gene_name'].tolist()

In [3]:
import scanpy as sc

In [4]:
context_A_data = sc.read_h5ad('../../data/controls/context_A.h5ad')
context_B_data = sc.read_h5ad('../../data/controls/context_B.h5ad')
context_C_data = sc.read_h5ad('../../data/controls/context_C.h5ad')

In [5]:
pert_counts = pd.read_csv('../../data/controls/pert_counts.csv')

In [6]:
target_genes = pert_counts['target_gene'].tolist()

In [7]:
contexts_and_data = {
    'A': context_A_data,
    'B': context_B_data,
    'C': context_C_data
}

In [8]:
target_genes_by_context = {c: target_genes for c in contexts_and_data}

## Train model

In [9]:
model_obj = Model(gene_names_list)

In [10]:
paths = list(Model.DEFAULT_TRAINING_PATHS)   # all four screens
model_obj = Model(gene_names_list)
model_obj.fit(paths, K=50, lam=0.01, effect_space="log2")
kd = model_obj.knockdown_from_training(paths)

In [11]:
del model_obj._Y
model_obj._dataset_cache.clear()

model_obj.write_submission(
    'pc_linreg_log2_K50_fidfix.h5ad',
    contexts_and_data,
    target_genes_by_context,
    knockdown_fraction=kd,
    alpha=1.0,
    rounding="stochastic",
    gain=1.5,
    cells_per_pert=400,   # required by the submission format
)

'pc_linreg_log2_K50_fidfix.h5ad'

In [ ]:
sub.write_h5ad('pc_linreg_log2.h5ad', compression='gzip')